# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Uzakov Islam
**Student ID:**
**Group:**
**Repository:** https://github.com/MegaZhaba/css4007-hw3

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> I used **Claude Code (Anthropic)** as an assistant for this homework: to set up the environment, write the notebook code (data loading, `rank_bm25` and EmbeddingGemma retrieval, prompt building, OpenAI calls), check the Part A arithmetic, and draft the written answers. Every number, chunk id, score and token count quoted in the written answers is copied from the outputs this notebook printed in its final top-to-bottom run, and I checked them against those outputs. No API key appears in any cell or output: keys are read from `.env`, which is git-ignored.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

Tokenisation: lowercase, split at spaces and hyphens, drop `.` and `?`, keep every word.

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** how, does, rag, retrieve, information (5 terms)

**TF, DF and IDF of each query term** (N = 3):

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = ln 7 = 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

Length normalisation `K(d) = k1·(1 − b + b·|d|/avgdl)`:
D1: 1.5·(0.25 + 0.75·9/8.3333) = 1.5·1.06 = 1.590;
D2: 1.5·(0.25 + 0.75·1.2) = 1.5·1.15 = 1.725;
D3: 1.5·(0.25 + 0.75·0.72) = 1.5·0.79 = 1.185.

**BM25 of each document:** every query term has f(t,d) = 0 in every document, so every term contributes
`idf · 0 · 2.5 / (0 + K(d)) = 0`.

> D1 = 1.9459·0 + 1.9459·0 + 1.9459·0 + 1.9459·0 + 1.9459·0 = **0**
> D2 = 0 + 0 + 0 + 0 + 0 = **0**  ("retrieves" ≠ "retrieve", "retrieval" ≠ "retrieve", "RAG" never appears spelled out as "rag")
> D3 = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 25 / 3 = **8.3333** (stemming does not change the number of tokens)

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | **2** | 0 | 1 | ln((3−1+0.5)/(1+0.5)) = ln(2.5/1.5) = **0.5108** |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

(`gener` now occurs in D1 and D2, but it is not a query term, so it does not score.)

**BM25 of each document**, worked (K(d) as in (a): D1 1.590, D2 1.725, D3 1.185):

> D1: only possible term is retriev, f = 0 → 0.5108 · 0 · 2.5 / (0 + 1.590) = **0**
> D2: retriev, f = 2 → 0.5108 · 2 · 2.5 / (2 + 1.725) = 0.5108 · 5 / 3.725 = 0.5108 · 1.3423 = **0.6857**; all other terms 0
> D3: retriev, f = 0 → **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a): none.** All three scores are exactly 0, so BM25 has no preference; any "first" is just the tie-break order (a library would return D1 because it comes first in the list). This is not an arithmetic error: BM25 only counts *exact token matches*, and no query token occurs in any document. "retrieve" ≠ "retrieves"/"retrieval", "information" occurs nowhere, and "RAG" is the abbreviation of "Retrieval-Augmented Generation", which BM25 cannot know. The idf of 1.9459 is high for every term only because each term has DF = 0, and multiplying it by tf = 0 still gives 0.
>
> **(b): D2.** Stemming maps "retrieve", "retrieves" and "retrieval" to the same token `retriev`, which occurs twice in D2 and nowhere else, giving D2 = 0.6857. D1 and D3 stay at 0. Stemming fixed the morphology mismatch, but "rag" and "inform" still match nothing; BM25 still cannot see that RAG = Retrieval-Augmented Generation.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = **0.9644**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = **0.78** | 0.9644 | √(0.49+0.01+0.16) = √0.66 = 0.8124 | 0.78 / (0.9644·0.8124) = 0.78 / 0.7835 = **0.9956** |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = **1.08** | 0.9644 | √(0.81+0.09+0.36) = √1.26 = 1.1225 | 1.08 / (0.9644·1.1225) = 1.08 / 1.0825 = **0.9977** |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = **0.34** | 0.9644 | √(0.01+0.64+0.04) = √0.69 = 0.8307 | 0.34 / (0.9644·0.8307) = 0.34 / 0.8011 = **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2** (0.9977), but only just ahead of D1 (0.9956). D3 (0.4244) points in a clearly different direction.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Against (a): no. BM25 (a) gave all three documents 0 and could not rank them at all, while cosine gives a full order D2 > D1 > D3. Against (b): the top document agrees (D2), but BM25 (b) still ties D1 and D3 at 0, while the embedding puts D1 almost level with D2 and far above D3. So cosine yields a graded ranking where BM25 yields one non-zero score.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 compares strings: the query words "how", "does", "RAG", "retrieve", "information" share no token with any document, so the score is 0 everywhere, and even with stemming only `retriev` connects to D2. An embedding model compares meaning: it can place "RAG" near "Retrieval-Augmented Generation", "retrieve" near "retrieves", and "information" near "documents" and "answer" in D2, and "answer questions" in D1 is semantically close to retrieving information (hence D1's 0.9956), even though no word is shared. D3 is about vectors/embeddings, a different topic, so it is far in both systems. Disagreement comes from abbreviations, morphology and paraphrase, which BM25 cannot bridge and embeddings can.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
# Arithmetic check for Part A (the working is written out above)
import math

K1, B = 1.5, 0.75

def bm25_by_hand(docs, query):
    N = len(docs)
    avgdl = sum(len(d) for d in docs) / N
    scores = []
    for d in docs:
        s = 0.0
        for t in query:
            n = sum(t in dd for dd in docs)
            idf = math.log((N - n + 0.5) / (n + 0.5))
            f = d.count(t)
            s += idf * f * (K1 + 1) / (f + K1 * (1 - B + B * len(d) / avgdl))
        scores.append(s)
    return avgdl, scores

raw = ["Large language models can generate text and answer questions.",
       "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
       "Embeddings represent text as numerical vectors."]
tok = lambda s: s.lower().replace("-", " ").replace(".", "").replace("?", "").split()
docs_a = [tok(d) for d in raw]
q_a = tok("How does RAG retrieve information?")
STEM = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
        "generate": "gener", "generating": "gener", "generation": "gener",
        "information": "inform", "does": "doe"}
stem = lambda toks: [STEM.get(t, t) for t in toks]
docs_b, q_b = [stem(d) for d in docs_a], stem(q_a)

for label, docs, q in [("(a) as written", docs_a, q_a), ("(b) stemmed", docs_b, q_b)]:
    avgdl, sc = bm25_by_hand(docs, q)
    print(label, "| query:", q, "| avgdl = %.4f" % avgdl)
    for i, (d, s) in enumerate(zip(docs, sc), 1):
        print(f"  D{i} |d|={len(d):2d}  BM25={s:.4f}  {d}")

Q = [0.8, 0.2, 0.5]
V = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
nq = math.sqrt(sum(x * x for x in Q))
print("\n|Query| = %.4f" % nq)
for k, v in V.items():
    dot = sum(a * b for a, b in zip(Q, v)); nd = math.sqrt(sum(x * x for x in v))
    print(f"{k}: dot={dot:.2f}  |D|={nd:.4f}  cos={dot / (nq * nd):.4f}")

(a) as written | query: ['how', 'does', 'rag', 'retrieve', 'information'] | avgdl = 8.3333
  D1 |d|= 9  BM25=0.0000  ['large', 'language', 'models', 'can', 'generate', 'text', 'and', 'answer', 'questions']
  D2 |d|=10  BM25=0.0000  ['retrieval', 'augmented', 'generation', 'retrieves', 'relevant', 'documents', 'before', 'generating', 'an', 'answer']
  D3 |d|= 6  BM25=0.0000  ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors']
(b) stemmed | query: ['how', 'doe', 'rag', 'retriev', 'inform'] | avgdl = 8.3333
  D1 |d|= 9  BM25=0.0000  ['large', 'language', 'models', 'can', 'gener', 'text', 'and', 'answer', 'questions']
  D2 |d|=10  BM25=0.6857  ['retriev', 'augmented', 'gener', 'retriev', 'relevant', 'documents', 'before', 'gener', 'an', 'answer']
  D3 |d|= 6  BM25=0.0000  ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors']

|Query| = 0.9644
D1: dot=0.78  |D|=0.8124  cos=0.9956
D2: dot=1.08  |D|=1.1225  cos=0.9977
D3: dot=0.34  |D|=0.8307  cos=0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json, os, re, warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")
from pathlib import Path
from dotenv import load_dotenv
import numpy as np

load_dotenv(Path(".env"))
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")), "| HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))

corpus = [json.loads(line) for line in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]
qa = json.load(open("data/qa.json", encoding="utf-8"))["questions"]
by_id = {c["id"]: c for c in corpus}
ids = [c["id"] for c in corpus]
texts = [c["text"] for c in corpus]

print(len(corpus), "chunks |", len(queries), "fixed queries |", len(qa), "QA questions")
from collections import Counter
print(Counter(c["lang"] for c in corpus))

OPENAI_API_KEY set: True | HF_TOKEN set: True
74 chunks | 8 fixed queries | 6 QA questions
Counter({'en': 71, 'ru': 2, 'kk': 1})


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> Lowercase, then the regex `\w+` with Unicode matching. So punctuation is dropped, hyphens split words (`GR-305` → `gr`, `305`; `need-based` → `need`, `based`), and Cyrillic/Kazakh words are kept as tokens. **No stemming and no stop-word removal**, so `is`, `the`, `do`, `can` count like any other word. `BM25Okapi(k1=1.5, b=0.75)`. The missing stop-word list turns out to matter: several BM25 failures below are driven by `is`, `the`, `can`, `do`.

In [3]:
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r"\w+", re.UNICODE)

def tokenize(text):
    # lowercase, Unicode \w+ (keeps Cyrillic/Kazakh letters, splits GR-305 into "gr" + "305"),
    # no stemming, no stop-word removal
    return TOKEN_RE.findall(text.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def bm25_top(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(-scores)[:k]
    return [(ids[i], float(scores[i])) for i in order]

def first_line(cid, width=70):
    c = by_id[cid]
    return (c["title"] + " — " + c["text"])[:width]

def show(results):
    for rank, (cid, s) in enumerate(results, 1):
        print(f"  {rank}. {cid}  {s:7.3f}  {first_line(cid)}")

print("Q-01 with BM25:"); show(bm25_top(queries[0]["text"]))

Q-01 with BM25:
  1. GO-08   15.825  Form GR-305: appeals — Form GR-305 is used to appeal a decision of the
  2. GO-12    8.195  How to appeal a decision — A student who disagrees with a decision may
  3. GO-06    8.070  Form GR-104: the application form — Form GR-104 is the application for


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("google/embeddinggemma-300m", token=os.getenv("HF_TOKEN"))  # float32, CPU
doc_emb = emb_model.encode_document(texts, convert_to_tensor=True)
print("document embeddings:", tuple(doc_emb.shape))

def emb_top(query, k=3):
    q = emb_model.encode_query(query, convert_to_tensor=True)
    sims = emb_model.similarity(q, doc_emb)[0].cpu().numpy()   # cosine
    order = np.argsort(-sims)[:k]
    return [(ids[i], float(sims[i])) for i in order]

print("Q-01 with EmbeddingGemma:"); show(emb_top(queries[0]["text"]))

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:  35%|███▍      | 109/314 [00:00<00:00, 986.80it/s]

Loading weights:  68%|██████▊   | 213/314 [00:00<00:00, 1009.12it/s]

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1363.00it/s]

document embeddings: (74, 768)
Q-01 with EmbeddingGemma:
  1. GO-08    0.696  Form GR-305: appeals — Form GR-305 is used to appeal a decision of the
  2. GO-12    0.461  How to appeal a decision — A student who disagrees with a decision may
  3. GO-23    0.450  Апелляция: кратко — Если студент не согласен с решением, он может пода


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What happens if a GR-210 is missing a signature? | GO-07 | **BM25.** The form code `GR-210` is an exact, rare string (`210` occurs in only two chunks), and `signature` occurs only in GO-07. An embedding model may blur GR-210 with the near-identical GR-104/GR-305 form chunks. |
| Q-10 | Can I keep my funding if I move to a different college? | GO-18 | **Embeddings.** It is a paraphrase: GO-18 says "transfers to another university loses it", while the query says "move", "college", "funding". BM25 has almost no word in common with the answer. |

In [5]:
# Written down before running anything below.
my_queries = [
    {"id": "Q-09", "text": "What happens if a GR-210 is missing a signature?",
     "relevant": ["GO-07"], "expected": "BM25"},
    {"id": "Q-10", "text": "Can I keep my funding if I move to a different college?",
     "relevant": ["GO-18"], "expected": "embeddings"},
]
all_queries = queries + my_queries
print(len(all_queries), "queries in total")

10 queries in total


## Results for all ten or more queries: top 3 from each retriever

In [6]:
results = {}
for q in all_queries:
    rb, re_ = bm25_top(q["text"]), emb_top(q["text"])
    hb = any(cid in q["relevant"] for cid, _ in rb)
    he = any(cid in q["relevant"] for cid, _ in re_)
    results[q["id"]] = {"bm25": rb, "emb": re_, "hit_bm25": hb, "hit_emb": he}
    print(f"{q['id']}: {q['text']}   relevant={q['relevant']}")
    print(f" BM25            hit@3={hb}"); show(rb)
    print(f" EmbeddingGemma  hit@3={he}"); show(re_)
    print()

print(f"{'Query':6} | {'BM25 top 3':26} | hit | {'Embedding top 3':26} | hit")
for q in all_queries:
    r = results[q["id"]]
    print(f"{q['id']:6} | {', '.join(c for c, _ in r['bm25']):26} | {'✓' if r['hit_bm25'] else '✗':3} | "
          f"{', '.join(c for c, _ in r['emb']):26} | {'✓' if r['hit_emb'] else '✗'}")
print("hit@3 total: BM25 %d/%d, EmbeddingGemma %d/%d" % (
    sum(r["hit_bm25"] for r in results.values()), len(results),
    sum(r["hit_emb"] for r in results.values()), len(results)))

Q-01: What is form GR-305 used for?   relevant=['GO-08', 'GO-12', 'GO-23']
 BM25            hit@3=True
  1. GO-08   15.825  Form GR-305: appeals — Form GR-305 is used to appeal a decision of the
  2. GO-12    8.195  How to appeal a decision — A student who disagrees with a decision may
  3. GO-06    8.070  Form GR-104: the application form — Form GR-104 is the application for
 EmbeddingGemma  hit@3=True
  1. GO-08    0.696  Form GR-305: appeals — Form GR-305 is used to appeal a decision of the
  2. GO-12    0.461  How to appeal a decision — A student who disagrees with a decision may
  3. GO-23    0.450  Апелляция: кратко — Если студент не согласен с решением, он может пода



Q-02: My family is poor. Is there financial help for students like me?   relevant=['GO-01', 'GO-02', 'GO-03']
 BM25            hit@3=False
  1. KZ-17    8.726  Lake Balkhash (part 2) — The lake currently covers about 16,400 km2 (6
  2. GO-20    8.311  Who cannot apply — The need-based grant is for full-time undergraduate
  3. KZ-25    7.511  Beshbarmak (part 2) — Beshbarmak is usually made from finely chopped b
 EmbeddingGemma  hit@3=True
  1. GO-01    0.445  About the need-based study grant 2026 — The need-based study grant 202
  2. GO-10    0.412  What counts as proof of income — Proof of income must accompany form G
  3. GO-20    0.409  Who cannot apply — The need-based grant is for full-time undergraduate



Q-03: Өтінішті қашанға дейін тапсыру керек?   relevant=['GO-09', 'GO-14']
 BM25            hit@3=True
  1. GO-14   10.698  Өтініш беру мерзімі және тәртібі — 2026 жылғы грантқа өтініштер 1 қырк
  2. KZ-01    0.000  Kazakhstan (part 1) — Kazakhstan, officially the Republic of Kazakhsta
  3. KZ-03    0.000  Kazakhstan (part 3) — Kazakhstan has been inhabited since the Paleolit
 EmbeddingGemma  hit@3=True
  1. GO-14    0.399  Өтініш беру мерзімі және тәртібі — 2026 жылғы грантқа өтініштер 1 қырк
  2. GO-09    0.350  Application deadline — Applications for the 2026 grant open on 1 Septe
  3. GO-11    0.301  When a document is missing — If a required document is missing, the Gr



Q-04: What is the minimum GPA for the grant?   relevant=['GO-02']
 BM25            hit@3=False
  1. GO-05   12.780  Archived: the 2025 scheme (no longer in force) — ARCHIVED — kept for r
  2. GO-17   10.518  Academic progress during the year — The grant continues only while the
  3. GO-19   10.136  Priority when the budget runs short — The grant budget is fixed each y
 EmbeddingGemma  hit@3=True
  1. GO-02    0.665  Who qualifies — An applicant qualifies for the 2026 grant when all thr
  2. GO-05    0.621  Archived: the 2025 scheme (no longer in force) — ARCHIVED — kept for r
  3. GO-17    0.613  Academic progress during the year — The grant continues only while the



Q-05: When was Astana renamed Nur-Sultan, and why?   relevant=['KZ-05']
 BM25            hit@3=True
  1. KZ-05   15.731  Astana — Astana is the capital city of Kazakhstan. With a population o
  2. KZ-06    6.443  Almaty (part 1) — Almaty is the most populous city in Kazakhstan, with
  3. KZ-11    5.866  Baikonur Cosmodrome (part 2) — In 1955, the Soviet Ministry of Defense
 EmbeddingGemma  hit@3=True
  1. KZ-05    0.651  Astana — Astana is the capital city of Kazakhstan. With a population o
  2. KZ-06    0.507  Almaty (part 1) — Almaty is the most populous city in Kazakhstan, with
  3. KZ-03    0.419  Kazakhstan (part 3) — Kazakhstan has been inhabited since the Paleolit



Q-06: Where did the Soviet Union test its nuclear weapons?   relevant=['KZ-37', 'KZ-38']
 BM25            hit@3=True
  1. KZ-37   19.090  Semipalatinsk Test Site (part 1) — The Semipalatinsk Test Site or Semi
  2. KZ-38   17.168  Semipalatinsk Test Site (part 2) — The Soviet Union conducted 456 nucl
  3. KZ-03    9.352  Kazakhstan (part 3) — Kazakhstan has been inhabited since the Paleolit
 EmbeddingGemma  hit@3=True
  1. KZ-37    0.596  Semipalatinsk Test Site (part 1) — The Semipalatinsk Test Site or Semi
  2. KZ-38    0.510  Semipalatinsk Test Site (part 2) — The Soviet Union conducted 456 nucl
  3. KZ-10    0.331  Baikonur Cosmodrome (part 1) — The Baikonur Cosmodrome is a spaceport 



Q-07: Can the grant be sent to my mother's card?   relevant=['GO-15']
 BM25            hit@3=False
  1. GO-21   14.652  Contacting the Grant Office — The Grant Office is in room 214 of the m
  2. GO-16    9.160  Combining the grant with other support — The need-based grant can be h
  3. GO-01    9.053  About the need-based study grant 2026 — The need-based study grant 202
 EmbeddingGemma  hit@3=True
  1. GO-15    0.422  Выплата гранта — Грант выплачивается двумя равными частями — в октябре
  2. GO-18    0.417  Academic leave and transfers — A student who goes on academic leave ke
  3. GO-01    0.411  About the need-based study grant 2026 — The need-based study grant 202



Q-08: If my grades drop in the autumn, do I still get paid in February?   relevant=['GO-17']
 BM25            hit@3=True
  1. GO-17   17.168  Academic progress during the year — The grant continues only while the
  2. GO-05   12.773  Archived: the 2025 scheme (no longer in force) — ARCHIVED — kept for r
  3. GO-04   11.932  How much the grant pays — The grant amount depends only on the income 
 EmbeddingGemma  hit@3=True
  1. GO-17    0.559  Academic progress during the year — The grant continues only while the
  2. GO-18    0.459  Academic leave and transfers — A student who goes on academic leave ke
  3. GO-15    0.453  Выплата гранта — Грант выплачивается двумя равными частями — в октябре



Q-09: What happens if a GR-210 is missing a signature?   relevant=['GO-07']
 BM25            hit@3=True
  1. GO-07   19.397  Form GR-210: the income declaration — Form GR-210 is the household inc
  2. GO-11   13.758  When a document is missing — If a required document is missing, the Gr
  3. GO-13   12.974  Changes in circumstances — A student whose household income changes du
 EmbeddingGemma  hit@3=True
  1. GO-07    0.619  Form GR-210: the income declaration — Form GR-210 is the household inc
  2. GO-10    0.426  What counts as proof of income — Proof of income must accompany form G
  3. GO-13    0.394  Changes in circumstances — A student whose household income changes du



Q-10: Can I keep my funding if I move to a different college?   relevant=['GO-18']
 BM25            hit@3=True
  1. GO-18    7.243  Academic leave and transfers — A student who goes on academic leave ke
  2. GO-21    6.919  Contacting the Grant Office — The Grant Office is in room 214 of the m
  3. GO-11    6.763  When a document is missing — If a required document is missing, the Gr
 EmbeddingGemma  hit@3=True
  1. GO-18    0.609  Academic leave and transfers — A student who goes on academic leave ke
  2. GO-17    0.459  Academic progress during the year — The grant continues only while the
  3. GO-13    0.412  Changes in circumstances — A student whose household income changes du

Query  | BM25 top 3                 | hit | Embedding top 3            | hit
Q-01   | GO-08, GO-12, GO-06        | ✓   | GO-08, GO-12, GO-23        | ✓
Q-02   | KZ-17, GO-20, KZ-25        | ✗   | GO-01, GO-10, GO-20        | ✓
Q-03   | GO-14, KZ-01, KZ-03        | ✓   | GO-14, GO-09, GO-11        | ✓
Q-04  

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-03 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-11, GO-13 | ✓ | GO-07, GO-10, GO-13 | ✓ |
| Q-10 | GO-18, GO-21, GO-11 | ✓ | GO-18, GO-17, GO-13 | ✓ |
| **hit@3 total** | | **7**/10 | | **10**/10 |

My predictions: Q-09 (BM25) — both hit it; BM25 was more decisive (GO-07 19.40 vs the next 13.76), but it did not "win" on hit@3. Q-10 (embeddings) — **my prediction was wrong on hit@3**: BM25 also put GO-18 first, but only through the single shared word `different` (4.04 of its 7.24) and with a tiny margin over GO-21 (6.92, matched on `can`, `to`, `a`). EmbeddingGemma separated it clearly (0.609 vs 0.459).

**Both retrievers agree:** which query, and why was it easy?

> **Q-06** "Where did the Soviet Union test its nuclear weapons?" Both return KZ-37 then KZ-38 (BM25 19.09 / 17.17, embeddings 0.596 / 0.510). It was easy because the answer chunks share the query's rare content words (`soviet`, `union`, `nuclear`) *and* are about exactly that topic, so lexical overlap and meaning point the same way. Q-01 and Q-05 are similar (the literal strings `GR-305` and `Astana`/`Nur-Sultan`).

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> On hit@3 BM25 never beat EmbeddingGemma (7/10 vs 10/10), so the closest cases are where BM25 ranked the answer more sharply because of an exact string. **Q-09**: the strings `210` and `signature` occur together only in GO-07, giving 19.40 against 13.76 for the next chunk; EmbeddingGemma also ranked GO-07 first (0.619) but its 2nd and 3rd places (GO-10 0.426, GO-13 0.394) are generic "income documents" chunks — it blurs GR-210 into "income paperwork". **Q-03**: BM25 found the Kazakh chunk GO-14 with 10.70 while *every other chunk scored 0.000*; it matched the exact Kazakh strings `дейін` and `керек`, which occur nowhere else. Embeddings ranked GO-14 first only by 0.399 vs 0.350.

**Embeddings did better:** which query, and what could BM25 not match?

> **Q-07** "Can the grant be sent to my mother's card?" The answer is only in **Russian** (GO-15: "перевод на счёт родителей … невозможен"), so BM25 has no shared token at all (GO-15 got 0 from every query term). Instead it ranked GO-21 first (14.65) on the words `can`, `sent` ("Questions can also be *sent*"), `the`, `be`, `to` — the contact-office chunk. EmbeddingGemma is multilingual and put GO-15 first (0.422). **Q-02** is a paraphrase failure: "family is poor … financial help" never appears in the handbook (it says "low-income household", "need-based grant"); BM25 put Lake Balkhash KZ-17 first on `is`, `there`, `like`, while embeddings put GO-01 first (0.445). **Q-04** is the trap: BM25 put the **archived** GO-05 first (12.78) because it literally contains "the minimum GPA for the grant" (`minimum` alone gives 4.57; the current rule GO-02 never says "minimum"), and GO-02 is not even in BM25's top 3. EmbeddingGemma put GO-02 first (0.665), but GO-05 was a close second (0.621) — read its title: *"Archived: the 2025 scheme (no longer in force)"*. Both retrievers rank it highly because it *is* about the minimum GPA; only reading the title tells you it is obsolete.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [7]:
import tiktoken
from openai import OpenAI

MODEL = "gpt-5.6-luna"
client = OpenAI()
enc = tiktoken.get_encoding("o200k_base")
ntok = lambda s: len(enc.encode(s))

SYSTEM = ("Answer the question using only the provided context.\n"
          "If the answer cannot be found in the context, say that you do not know.\n"
          "Cite the context blocks you used, like [1] or [2][3].")

def context_block(hits):
    return "\n\n".join(f"[{i}] ({cid}) {by_id[cid]['title']}\n{by_id[cid]['text']}"
                       for i, (cid, _) in enumerate(hits, 1))

def build_messages(question, hits):
    if hits is None:                       # LLM only: the bare question
        return [{"role": "user", "content": question}]
    user = f"Context:\n\n{context_block(hits)}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def ask(messages):
    r = client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content.strip(), r.usage

RETRIEVERS = {"LLM only": lambda q, k: None, "BM25 RAG": bm25_top, "Embedding RAG": emb_top}

## C1 — the complete prompt, printed before it is sent

In [8]:
c1 = next(q for q in qa if q["id"] == "C-02")
hits = emb_top(c1["question"], 3)
msgs = build_messages(c1["question"], hits)

print("=" * 30, "EXACTLY WHAT IS SENT", "=" * 30)
for m in msgs:
    print(f"--- role: {m['role']} ---")
    print(m["content"])
print("=" * 80)

ctx = context_block(hits)
t_sys, t_q, t_ctx = ntok(SYSTEM), ntok(c1["question"]), ntok(ctx)
t_total = sum(ntok(m["content"]) for m in msgs)
print(f"system instructions : {t_sys}")
print(f"question            : {t_q}")
print(f"retrieved context   : {t_ctx}")
print(f"wrapper text ('Context:', 'Question:', blank lines): {t_total - t_sys - t_q - t_ctx}")
print(f"TOTAL content tokens (tiktoken o200k_base): {t_total}")
print(f"chunks: {len(hits)}")
for cid, s in hits:
    print(f"  {cid}  score={s:.3f}  tokens={ntok(by_id[cid]['text'])}  chars={len(by_id[cid]['text'])}")
print(f"context size: {t_ctx} tokens / {len(ctx)} characters")

answer, usage = ask(msgs)
print("\nusage.prompt_tokens     :", usage.prompt_tokens)
print("usage.completion_tokens :", usage.completion_tokens)
print("difference (usage − tiktoken):", usage.prompt_tokens - t_total)
print("\nANSWER:\n" + answer)

============================== EXACTLY WHAT IS SENT ==============================
--- role: system ---
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].
--- role: user ---
Context:

[1] (GO-08) Form GR-305: appeals
Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

[2] (GO-12) How to appeal a decision
A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the


usage.prompt_tokens     : 356
usage.completion_tokens : 31
difference (usage − tiktoken): 10

ANSWER:
You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**. [2]


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 21 |
| Retrieved context | 277 |
| (wrapper: `Context:`, `Question:`, blank lines) | 4 |
| **Total input** | **346** |
| **`usage.prompt_tokens` from the response** | **356** |

- Number of retrieved chunks: 3 (GO-08, GO-12, GO-11 from EmbeddingGemma, for C-02)
- Tokens per chunk (chunk text only): GO-08 68, GO-12 88, GO-11 78 (= 234; the other 43 of the 277 context tokens are the `[1] (GO-08) Form GR-305: appeals` headers and separators)
- Total context size (tokens / characters): 277 tokens / 1263 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> 356 − 346 = **10 tokens**. tiktoken counts only the text of my strings. The API wraps the messages in a chat template before the model sees them: each message gets special role/separator tokens (start-of-message, the role name `system`/`user`, end-of-message), and a few tokens are added to prime the assistant's reply. Two messages × ~3–4 tokens plus ~3 priming tokens ≈ 10. It is the same gap as in HW1: the bill counts the formatted conversation, not the raw text.

**What exactly does retrieval add to the model's input?**

> Literally: **277 tokens of text** — three chunks copied verbatim from `corpus.jsonl`, each with a number `[1]`–`[3]`, its id and title, pasted into the user message between `Context:` and `Question:`. That is all. Retrieval does not give the model the corpus, an index, vectors or scores; the model never sees GO-23 or any other chunk that was not selected, and has no idea that chunks were "retrieved" except that my prompt says so. Here retrieval also added an irrelevant chunk (GO-11, about missing documents), which costs 78 tokens and is simply ignored by the answer.

## C2 — three systems, six questions

In [9]:
c2 = {}
for q in qa:
    c2[q["id"]] = {}
    print("=" * 90)
    print(f"{q['id']}: {q['question']}")
    print(f"REFERENCE: {q['reference']}")
    for name, retr in RETRIEVERS.items():
        hits = retr(q["question"], 3)
        answer, _ = ask(build_messages(q["question"], hits))
        c2[q["id"]][name] = {"hits": hits, "answer": answer}
        sent = "" if hits is None else "  context: " + ", ".join(f"{c} ({s:.2f})" for c, s in hits)
        print(f"\n--- {name}{sent}\n{answer}")

C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.



--- LLM only
Which student funding scheme or institution do you mean? “Income band 2” is used by several programs, and the amount and payment schedule vary. Please share the name, country, or a link to the relevant table.



--- BM25 RAG  context: GO-04 (20.22), GO-02 (18.74), GO-01 (17.24)
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]



--- Embedding RAG  context: GO-04 (0.58), GO-03 (0.48), GO-15 (0.45)
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]
C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.



--- LLM only
The deadline and appeal form depend on **which grant, agency, and country/jurisdiction** issued the refusal. The refusal decision letter should state:

- the deadline for requesting a review or appeal;
- whether it is called a **reconsideration**, **review**, or **appeal**; and
- the required form or online process.

Please tell me the **grant name/agency and jurisdiction**, or paste the relevant part of the refusal letter (remove personal information). Don’t wait for clarification if the deadline is near—check the letter and the agency’s appeals webpage immediately, as deadlines can be as short as 10–30 days.


...


--- BM25 RAG  context: KZ-44 (17.57), KZ-11 (13.76), KZ-07 (12.26)
Narxoz University was founded in 1963 as the Almaty Institute of National Economy. [1]



--- Embedding RAG  context: KZ-44 (0.66), KZ-05 (0.27), KZ-14 (0.25)
Narxoz University was founded in 1963 as the **Almaty Institute of National Economy**. [1]
C-06: I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.



--- LLM only
The amount depends heavily on your **country, university, program, enrollment status, and financial circumstances**. If you mean the **United States**:

- **Federal Pell Grants:** Generally **not available for regular master’s degrees**. Pell is mainly for undergraduate students. An exception may apply if your program is an eligible undergraduate-level teacher-certification program.
- **TEACH Grant:** Available only for certain education programs and service commitments; the maximum is typically around **$4,000 per year before any required reductions**, but eligibility is limited.
- **Federal Supplemental Educational Opportunity Grant (FSEOG):** Usually for undergraduates, so master’s students generally do not qualify.
- **University and state grants:** These are often the main need-based grant sources for master’s students. Awards can range from a few hundred dollars to covering a substantial portion of tuition, depending on the school.
- **Federal aid commonly available


--- BM25 RAG  context: GO-16 (17.00), GO-01 (14.17), GO-20 (13.72)
I do not know how much you can get. The need-based grant is only for full-time undergraduate students; master's students are covered by a separate scheme and should ask their department for its rules. [3]



--- Embedding RAG  context: GO-20 (0.67), GO-16 (0.62), GO-01 (0.55)
I do not know the amount. The need-based grant is only for full-time undergraduate students; master’s students are covered by a separate scheme and should ask their department for its rules. [1]


Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asks "which scheme do you mean?", no amount | ✓ 150,000 tenge, two equal instalments, Oct + Feb [1] | ✓ 150,000 tenge, 75,000 in Oct + 75,000 in Feb [1] |
| C-02 | ✗ "depends on the grant/jurisdiction", no form | ✓ 14 calendar days from the decision letter, GR-305 [2][3] | ✓ 14 calendar days from the decision letter, GR-305 [2] |
| C-03 | ✗ asks which grant; mentions Pell Grant | ✓ 2.67 on a 4.0 scale [2] | ✓ 2.67 on a 4.0 scale [1] |
| C-04 | ✓ decree 26 Oct 2017, target 2025, now 2031 | ✓ Oct 2017, initially 2025, now 2031 [1][2] | ✓ Oct 2017, now by 2031 [1][2] |
| C-05 | ✓ 1963, "Alma-Ata Institute of National Economy" | ✓ 1963, Almaty Institute of National Economy [1] | ✓ 1963, Almaty Institute of National Economy [1] |
| C-06 | ✗ long US answer (Pell, TEACH ~$4,000, FAFSA) — invented context | ✓ "I do not know"; separate scheme, ask department [3] | ✓ "I do not know"; separate scheme, ask department [1] |
| **agrees with `reference`** | **2**/6 | **6**/6 | **6**/6 |

Notes on marking: C-05 LLM-only says "Alma-Ata" — the old name of Almaty, so it is the same institute, marked ✓. C-04: neither RAG answer mentions that the 2031 date was set in January 2021, and the Embedding answer drops the original 2025 target, but the two facts asked (when decided, when due to finish) are right, so ✓. C-01 Embedding RAG adds "75,000 tenge" — not written in GO-04, but it is the arithmetic of "150,000 in two equal instalments", so it is correct, not invented. The LLM only system got right exactly the two questions whose answers are on Wikipedia (C-04, C-05), and none of the four handbook questions.

## C3 — question 1 of 3: C-03 (minimum GPA, and the archived rule)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 15.799 · GO-02 14.059 · GO-06 13.852. Rank 1, GO-16 ("Combining the grant with other support"), does not mention GPA at all; it won on `need` (4.61), `based` (3.99), `grant`, `the`, `is`, because "need-based grant" occurs in it three times. GO-02 came second through `qualify` (4.02) and `gpa` (2.62).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-02 0.629 · GO-20 0.569 · GO-05 0.545 — the right chunk first, but the **archived** 2025 rule GO-05 ("minimum GPA … was 3.0") is in the context too.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: `[1] (GO-16) Combining the grant…`, `[2] (GO-02) Who qualifies` (contains "cumulative GPA is at least 2.67 on the 4.0 scale"), `[3] (GO-06) Form GR-104…`.
> Embedding RAG: `[1] (GO-02) Who qualifies`, `[2] (GO-20) Who cannot apply`, `[3] (GO-05) Archived: the 2025 scheme (no longer in force)` (contains "minimum GPA for the grant was 3.0"). Full text is printed by the cell below.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Both RAG answers: "The minimum GPA is **2.67 on a 4.0 scale**", citing GO-02 ([2] for BM25, [1] for embeddings). Compared with LLM only ("Which need-based grant do you mean? … Pell Grant…"), retrieval made the answer **correct, specific and grounded**. In the Embedding case the model saw both 2.67 and the archived 3.0 and correctly picked the one marked current.
>
> **Where RAG got worse than I expected** (extra run at the bottom of the cell): asked with Q-04's wording, *"What is the minimum GPA for the grant?"*, BM25's top 3 was GO-05 12.780 · GO-17 10.518 · GO-19 10.136 — **GO-02 was not retrieved**. The model answered: *"I do not know the current minimum GPA. The only stated threshold is 3.0 under the archived 2025 scheme…"*. The top-K run (final question 8) shows the same thing for BM25 at K=1: only GO-16 was sent and the answer was "I do not know."

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> In the C2 runs nothing was missed. In the Q-04-wording run the fact "2.67" was missed, and it was a **retrieval failure**: "at least 2.67" is in GO-02 only, and GO-02 was not in the context (GO-05, GO-17, GO-19 were). Generation actually behaved well — it saw "ARCHIVED … do not apply to any application made for the 2026 academic year" and refused to present 3.0 as the answer instead of hallucinating. The cause is lexical: GO-05 contains the query's exact words "minimum GPA for the grant", while GO-02 phrases the rule as "cumulative GPA is at least 2.67" and never says "minimum".

In [10]:
qid = "C-03"
q = next(x for x in qa if x["id"] == qid)
print(q["question"]); print("REFERENCE:", q["reference"], "| relevant:", q["relevant"])
for name in ["BM25 RAG", "Embedding RAG"]:
    rec = c2[qid][name]
    print(f"\n##### {name} retrieved:")
    show(rec["hits"])
    print(f"\n----- context blocks sent ({name}) -----")
    print(context_block(rec["hits"]))
    print(f"\n----- answer ({name}) -----\n{rec['answer']}")
print("\n##### LLM only answer:\n" + c2[qid]["LLM only"]["answer"])

# Extra: the same question in Q-04's wording, with BM25's top 3 sent to the model
q04 = next(x for x in queries if x["id"] == "Q-04")["text"]
hits = bm25_top(q04, 3)
print(f"\n##### Extra — BM25 for Q-04 wording: {q04!r}")
show(hits)
print("answer:", ask(build_messages(q04, hits))[0])

What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.) | relevant: ['GO-02']

##### BM25 RAG retrieved:
  1. GO-16   15.799  Combining the grant with other support — The need-based grant can be h
  2. GO-02   14.059  Who qualifies — An applicant qualifies for the 2026 grant when all thr
  3. GO-06   13.852  Form GR-104: the application form — Form GR-104 is the application for

----- context blocks sent (BM25 RAG) -----
[1] (GO-16) Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] (GO-02) Who qualifies
An applicant qualifies for the 2026

answer: I do not know the current minimum GPA. The only stated threshold is 3.0 under the archived 2025 scheme, which no longer applies to 2026 applications. [1]


## C3 — question 2 of 3: C-06 (master's student — no answer in the corpus)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 16.996 · GO-01 14.173 · GO-20 13.720 — the relevant chunk GO-20 only third; GO-16 again wins on the repeated phrase "need-based grant".

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-20 0.666 · GO-16 0.616 · GO-01 0.554 — the same three chunks, with GO-20 first.

**3. What was sent to the model** (the context blocks):

> Both systems sent GO-20 ("Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules"), GO-16 (400,000 tenge cap) and GO-01 (grant is for full-time undergraduates). Only the order differed: BM25 [1] GO-16 [2] GO-01 [3] GO-20; embeddings [1] GO-20 [2] GO-16 [3] GO-01.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> This is the clearest win for RAG. LLM only answered a different question: a long US answer about Pell Grants, the TEACH Grant ("around **$4,000 per year**"), FSEOG and FAFSA — none of it from this university. Both RAG systems said **"I do not know"** the amount, and explained why: the need-based grant is only for full-time undergraduates, master's students have a separate scheme, ask the department (citing GO-20). That is the only correct response. Retrieval made the answer *grounded*, and the system prompt's "say that you do not know" did the rest. The context also contained a tempting number — "capped at 400,000 tenge" in GO-16 — and neither RAG answer used it.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed by RAG: there is no amount to find, and both answers correctly said so. The failure is in **LLM only**, and it is a **generation problem in the sense of "no grounding"**: with no context, the model filled the gap from its training data (US federal aid) instead of saying it does not know. Retrieval was imperfect for BM25 (GO-20 only at rank 3 with 13.720, below the off-topic GO-16 16.996), but with K=3 it still made it into the context, so it did no harm here.

In [11]:
qid = "C-06"
q = next(x for x in qa if x["id"] == qid)
print(q["question"]); print("REFERENCE:", q["reference"], "| relevant:", q["relevant"])
for name in ["BM25 RAG", "Embedding RAG"]:
    rec = c2[qid][name]
    print(f"\n##### {name} retrieved:")
    show(rec["hits"])
    print(f"\n----- context blocks sent ({name}) -----")
    print(context_block(rec["hits"]))
    print(f"\n----- answer ({name}) -----\n{rec['answer']}")
print("\n##### LLM only answer:\n" + c2[qid]["LLM only"]["answer"])

I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe. | relevant: ['GO-20']

##### BM25 RAG retrieved:
  1. GO-16   16.996  Combining the grant with other support — The need-based grant can be h
  2. GO-01   14.173  About the need-based study grant 2026 — The need-based study grant 202
  3. GO-20   13.720  Who cannot apply — The need-based grant is for full-time undergraduate

----- context blocks sent (BM25 RAG) -----
[1] (GO-16) Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] (GO-01) About the need-based study grant 2026
T

## C3 — question 3 of 3: C-02 (appeal deadline and form; BM25 sends the archived chunk)

**1. What BM25 retrieved** (top 3, ids and scores):

> **GO-05 17.241** · GO-08 14.529 · GO-12 13.573. The archived 2025 scheme came **first**, although it says nothing about appeals. It won on `do` (4.57 twice — "do not apply" matches "how long *do* I … which form *do* I use"), `application` (2.98), `was`, `grant`, `and`, `to`: function words, because I did not remove stop words.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-08 0.603 · GO-12 0.559 · GO-11 0.534 — both appeal chunks first, then GO-11 ("When a document is missing", which talks about decisions and deadlines). The Russian GO-23 (0.450 in Q-01) did not make it.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: `[1] (GO-05) Archived: the 2025 scheme…`, `[2] (GO-08) Form GR-305: appeals`, `[3] (GO-12) How to appeal a decision` ("may appeal within 14 calendar days of the date on the decision letter, using form GR-305").
> Embedding RAG: `[1] (GO-08)`, `[2] (GO-12)`, `[3] (GO-11)`. This is the same prompt that is printed in full in C1.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Both RAG systems: "You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**" (BM25 cites [2][3], embeddings cite [2]). LLM only: "The deadline and appeal form depend on which grant, agency, and country… deadlines can be as short as 10–30 days" — no form, no number. Retrieval made the answer **specific and correct**; the form code GR-305 cannot come from anywhere but the context.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing in the final answers was missed, but this is where retrieval was worse than expected: BM25's top chunk was the irrelevant archived GO-05 — a **retrieval error** (caused by stop words `do`, `was`, `and`), which generation compensated for by ignoring block [1] and citing only [2][3]. With K=1 BM25 would have sent *only* GO-05 and the answer could not have contained GR-305 or 14 days. Embedding retrieval also sent one irrelevant chunk (GO-11, "10 working days to upload it"); a careless generator could have mixed up "10 working days" with the 14-calendar-day appeal limit, but this one did not. One more retrieval note: neither system's context contained the Russian GO-23, which states the same rule — it was not needed here, because GO-12 has it in English.

In [12]:
qid = "C-02"
q = next(x for x in qa if x["id"] == qid)
print(q["question"]); print("REFERENCE:", q["reference"], "| relevant:", q["relevant"])
for name in ["BM25 RAG", "Embedding RAG"]:
    rec = c2[qid][name]
    print(f"\n##### {name} retrieved:")
    show(rec["hits"])
    print(f"\n----- context blocks sent ({name}) -----")
    print(context_block(rec["hits"]))
    print(f"\n----- answer ({name}) -----\n{rec['answer']}")
print("\n##### LLM only answer:\n" + c2[qid]["LLM only"]["answer"])

My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305. | relevant: ['GO-08', 'GO-12', 'GO-23']

##### BM25 RAG retrieved:
  1. GO-05   17.241  Archived: the 2025 scheme (no longer in force) — ARCHIVED — kept for r
  2. GO-08   14.529  Form GR-305: appeals — Form GR-305 is used to appeal a decision of the
  3. GO-12   13.573  How to appeal a decision — A student who disagrees with a decision may

----- context blocks sent (BM25 RAG) -----
[1] (GO-05) Archived: the 2025 scheme (no longer in force)
ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

[2] (GO-08) Form GR-305: appeals
Form GR-305

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 scores a chunk by counting exact token matches with the query, weighted by how rare each token is (IDF) and normalised for chunk length. Embedding retrieval turns query and chunk into dense vectors with a neural model and ranks by the angle between them, so it matches meaning rather than strings. In my runs BM25 gave GO-15 nothing for Q-07 (no shared token, the chunk is Russian), while EmbeddingGemma put it first.

**2. When can BM25 perform better than embeddings?**

> When the query contains a rare exact string: a form code, a number, a name, a rare word in any language. For Q-09 (`GR-210`, `signature`) BM25 separated GO-07 by 19.40 vs 13.76, and for the Kazakh Q-03 every chunk except GO-14 scored exactly 0. It is also cheaper, needs no model, and is easy to explain (I could list the exact tokens behind every score).

**3. When can embeddings perform better than BM25?**

> When the query is a paraphrase, uses synonyms, or is in a different language from the answer. In my runs: Q-02 ("family is poor … financial help" vs "low-income household … need-based grant"), Q-07 (English question, Russian answer GO-15), Q-04 (the current rule GO-02 never says "minimum"). EmbeddingGemma hit all three; BM25 missed all three, sometimes ranking chunks that matched only stop words (KZ-17 on `is`, `there`, `like`).

**4. Why is cosine similarity used with embeddings?**

> Cosine compares the *direction* of two vectors and ignores their length, and in an embedding space the direction encodes the meaning while the length mostly reflects things like text length or word frequency. That makes a short query comparable with a longer chunk. It is also bounded (−1…1), and for normalised vectors it equals the dot product, so it is cheap; `model.similarity` in sentence-transformers uses it by default. In Part A, D1 and D2 had very different dot products (0.78 vs 1.08) but almost the same cosine (0.9956 vs 0.9977), because they point in nearly the same direction.

**5. What information is actually sent to an LLM in a RAG system?**

> Only text: a system message, the retrieved chunks pasted in as numbered blocks, and the question. In C1 that was 44 + 277 + 21 (+4 wrapper) = 346 tiktoken tokens, billed as 356 `prompt_tokens`. No vectors, scores, index or other documents are sent.

**6. Does the LLM search the original documents itself?**

> No. The model never opens `corpus.jsonl`; my code searched it, picked three chunks and pasted them into the prompt. The model just continues that prompt. In the BM25 Q-04-wording run it said "I do not know the current minimum GPA" even though GO-02 with "2.67" was in the corpus — it can only use what was sent.

**7. What happens if retrieval returns an irrelevant chunk?**

> It costs tokens (GO-11 cost 78 tokens in C1) and it is a risk: the model may ignore it, or use it. In my runs the model mostly ignored irrelevant chunks (BM25 sent the archived GO-05 first for C-02 and the answer cited only [2][3]). But if the irrelevant chunks are *all* that was sent, the answer fails: BM25 at K=1 for C-03 sent only GO-16 and the answer was "I do not know"; with the Q-04 wording it sent GO-05/GO-17/GO-19 and the model could only report the archived 3.0. A weaker or less instructed model could have answered "3.0" outright.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> On C-03 (cell below):
>
> | K | BM25 chunks | BM25 answer | Embedding chunks | Embedding answer | prompt_tokens (BM25 / Emb) |
> |---|---|---|---|---|---|
> | 1 | GO-16 | "I do not know." ✗ | GO-02 | 2.67 ✓ | 158 / 185 |
> | 3 | GO-16, GO-02, GO-06 | 2.67 ✓ | GO-02, GO-20, GO-05 | 2.67 ✓ | 371 / 361 |
> | 8 | … + GO-05, GO-01, GO-11, GO-03, GO-17 | 2.67 ✓ | … + GO-16, GO-01, GO-06, GO-19, GO-17 | 2.67 ✓ | 900 / 856 |
>
> A larger K **improves recall**: for BM25, K=1 missed GO-02 and K=3 recovered it. But it also **adds noise and cost**: at K=8 the prompt is ~2.4× larger than at K=3 and contains the archived GO-05 (3.0) and GO-17 (also "2.67", but about the February instalment) next to the real rule, so the model has more chances to pick the wrong number or merge rules. Here the model resisted, so K=8 only cost tokens, but nothing improved past K=3.

In [13]:
# Top-K experiment on one question (C-03), both retrievers
qk = next(x for x in qa if x["id"] == "C-03")
for name, retr in [("BM25", bm25_top), ("EmbeddingGemma", emb_top)]:
    print("=====", name)
    for k in (1, 3, 8):
        hits = retr(qk["question"], k)
        answer, usage = ask(build_messages(qk["question"], hits))
        print(f"K={k}: {[c for c, _ in hits]}  prompt_tokens={usage.prompt_tokens}")
        print("   ", answer.replace("\n", " "))
    print()

===== BM25


K=1: ['GO-16']  prompt_tokens=158
    I do not know.


K=3: ['GO-16', 'GO-02', 'GO-06']  prompt_tokens=371
    The minimum GPA is **2.67 on a 4.0 scale**. [2]


K=8: ['GO-16', 'GO-02', 'GO-06', 'GO-05', 'GO-01', 'GO-11', 'GO-03', 'GO-17']  prompt_tokens=900
    The minimum GPA needed to qualify is **2.67 on a 4.0 scale**. [2]

===== EmbeddingGemma


K=1: ['GO-02']  prompt_tokens=185
    The minimum GPA needed is **2.67 on a 4.0 scale**. [1]


K=3: ['GO-02', 'GO-20', 'GO-05']  prompt_tokens=361
    The minimum GPA needed is **2.67 on a 4.0 scale**. [1]


K=8: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']  prompt_tokens=856
    The minimum GPA needed to qualify is **2.67 on a 4.0 scale**. [1]



**9. Is a vector database required to build a RAG system?**

> No. This notebook stores the 74 document vectors in a single in-memory tensor (`doc_emb`, shape 74 × 768) and finds the top 3 by computing cosine against all of them with `model.similarity` and `np.argsort`. BM25 needs no vectors at all. A vector database (FAISS, pgvector, Qdrant…) only becomes useful when there are millions of chunks, for approximate search, persistence and filtering.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> **EmbeddingGemma**, on retrieval: hit@3 **10/10** vs **7/10** for BM25. BM25 missed Q-02 (paraphrase), Q-04 (archived GO-05 first, GO-02 absent) and Q-07 (answer only in Russian), and put an irrelevant chunk first in C-02 and C-03 (GO-05, GO-16). On end-to-end answers the two are tied: C2 agreement **6/6 for BM25 RAG and 6/6 for Embedding RAG** (vs 2/6 for LLM only), because the C2 questions were long enough for BM25 to get the right chunk into the top 3 somewhere, and the model ignored the noise. The difference shows up at K=1 and with shorter wording (Q-04), where BM25 RAG answered "I do not know" and Embedding RAG still answered 2.67. BM25 remained the sharper tool for exact codes (Q-09) and the Kazakh query (Q-03).

---
# Conclusion

> Next I would build **hybrid retrieval with metadata filtering**: run BM25 and EmbeddingGemma in parallel and merge their rankings (e.g. reciprocal rank fusion), so that exact strings like `GR-210` and Kazakh words get BM25's precision while paraphrases and the Russian chunks get the embedding model's recall. Before both, I would add a stop-word list (several BM25 failures came from `is`, `the`, `do`, `can`) and **drop chunks marked archived** (GO-05) at indexing time, or tag them `status: archived` and filter them out — retrieval ranked GO-05 in the top 3 for Q-04, Q-08, C-02 and C-03, and the only thing that kept it out of the answers was the model reading the word "ARCHIVED". Finally I would keep K around 3 and log the exact prompt for every answer, since this homework showed that the fastest way to tell a retrieval failure from a generation failure is to read what was sent.